# witness-generation-bench — notebook de reproduction

ce notebook reprend, dans l'ordre, les étapes vérifiées pour le benchmark et le RAG.
tu peux l'exécuter cellule par cellule (recommandé la première fois), ou "exécuter tout".

**avant de commencer :** active un GPU (Runtime -> Modifier le type d'exécution -> GPU),
sinon les générations seront très lentes.

## 1. installation et récupération du code

In [ ]:
!git clone https://github.com/zeinmil/witness-generation-bench.git
%cd witness-generation-bench
!pip install -q -r requirements.txt

## 2. le corpus de schémas (jsonschemabench)

nécessaire pour la partie benchmark. si tu veux juste tester le RAG, tu peux sauter cette partie.

In [ ]:
import os
if not os.path.exists("/content/jsonschemabench"):
    !git clone https://github.com/guidance-ai/jsonschemabench.git /content/jsonschemabench
else:
    print("déjà cloné")

print("collections dispo :", os.listdir("/content/jsonschemabench/data/"))

## 3. config — modèle utilisé

**pour changer de modèle, modifie seulement les deux lignes ci-dessous.**
le reste du notebook s'adapte automatiquement (limite de tokens, gpu/cpu).

In [ ]:
import torch

MODEL_NAME = "microsoft/phi-2"          # <- change ça pour un autre modèle
MAX_CONTEXT_TOKENS = 2048                # <- fenêtre de contexte du modèle choisi
MAX_NEW_TOKENS = 200
MARGE_SECURITE = 48

MAX_INPUT_TOKENS = MAX_CONTEXT_TOKENS - MAX_NEW_TOKENS - MARGE_SECURITE
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"modèle : {MODEL_NAME} | device : {DEVICE} | max input tokens : {MAX_INPUT_TOKENS}")

## 4. fonctions communes (benchmark)

In [ ]:
import json
import jsonschema
from transformers import AutoModelForCausalLM, AutoTokenizer

INSTRUCTION = (
    "You generate JSON data. Given a JSON Schema, output ONE JSON instance "
    "that satisfies it.\nOutput only the JSON instance. Do not repeat the schema.\n"
)

EXEMPLES_FEWSHOT = [
    ('{"type":"object","not":{"required":["deprecated"]},"properties":{"title":{"type":"string"}}}',
     '{"title":"hello"}'),
    ('{"type":"array","not":{"contains":{"const":"zzz"}}}',
     '["alpha","beta"]'),
    ('{"type":"string","not":{"pattern":"^xx_"}}',
     '"kappa"'),
]

def contient_not(obj):
    """recherche récursive du mot-clé 'not' à n'importe quel niveau du schéma"""
    if isinstance(obj, dict):
        return "not" in obj or any(contient_not(v) for v in obj.values())
    if isinstance(obj, list):
        return any(contient_not(x) for x in obj)
    return False

def extraire_premier_json(texte):
    """extrait le premier objet json {...} complet d'un texte généré par le modèle"""
    d = texte.find("{")
    if d == -1:
        return None
    profondeur, en_chaine, echap = 0, False, False
    for i in range(d, len(texte)):
        c = texte[i]
        if en_chaine:
            if echap: echap = False
            elif c == "\\": echap = True
            elif c == '"': en_chaine = False
        else:
            if c == '"': en_chaine = True
            elif c == "{": profondeur += 1
            elif c == "}":
                profondeur -= 1
                if profondeur == 0:
                    return texte[d:i+1]
    return None

def construire_prompt(schema_text, condition):
    if condition == "zero-shot":
        return INSTRUCTION + f"\nSchema:\n{schema_text}\nInstance:\n"
    prompt = INSTRUCTION
    for sch, inst in EXEMPLES_FEWSHOT:
        prompt += f"\nSchema:\n{sch}\nInstance:\n{inst}\n"
    return prompt + f"\nSchema:\n{schema_text}\nInstance:\n"

print("fonctions chargées")

## 5. charger le modèle

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE, device_map=DEVICE)
print(f"{MODEL_NAME} chargé sur", model.device)

## 6. génération + validation sur un schéma

In [ ]:
def generer_et_valider(schema, condition, max_input_tokens=MAX_INPUT_TOKENS, max_new_tokens=MAX_NEW_TOKENS):
    schema_text = json.dumps(schema)
    prompt = construire_prompt(schema_text, condition)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_tok = inputs["input_ids"].shape[1]
    if n_tok > max_input_tokens:
        return {"instance": None, "valide": False, "erreur": f"overflow ({n_tok} tokens)"}

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    genere = tokenizer.decode(out[0][n_tok:], skip_special_tokens=True)

    candidat = extraire_premier_json(genere)
    if candidat is None:
        return {"instance": genere, "valide": False, "erreur": "tronqué / aucun objet json complet"}
    try:
        instance = json.loads(candidat)
        jsonschema.validate(instance, schema)
        return {"instance": instance, "valide": True, "erreur": None}
    except json.JSONDecodeError as e:
        return {"instance": candidat, "valide": False, "erreur": f"json mal formé : {e}"}
    except jsonschema.ValidationError as e:
        return {"instance": instance, "valide": False, "erreur": f"invalide : {e.message}"}

# on prend les schémas "not" de Github_easy pour un premier test
base = "/content/jsonschemabench/data/Github_easy/"
schemas_not = [n for n in os.listdir(base) if n.endswith(".json") and contient_not(json.load(open(base+n)))]
print(f"{len(schemas_not)} schémas avec 'not' trouvés dans Github_easy")

schema_test = json.load(open(base + schemas_not[0]))
resultat = generer_et_valider(schema_test, condition="zero-shot")
print("valide :", resultat["valide"])
print("erreur :", resultat["erreur"])
print("instance :", resultat["instance"])

## 7. lancer le benchmark complet sur une collection

décommente la ligne ci-dessous pour lancer sur toute une collection (peut prendre du temps).

In [ ]:
resultats = []
for fichier in schemas_not:
    schema = json.load(open(base + fichier))
    r = generer_et_valider(schema, condition="zero-shot")
    r["fichier"] = fichier
    resultats.append(r)
    print(f"{fichier:30s} -> {'VALIDE' if r['valide'] else r['erreur']}")

n_valides = sum(r["valide"] for r in resultats)
print(f"\nscore zero-shot : {n_valides}/{len(resultats)}")

## 8. étude d'ablation (retirer le "not" et comparer)

In [ ]:
import copy

def retirer_not(obj):
    """copie du schéma sans aucune clé "not", à n'importe quel niveau"""
    def _rec(o):
        if isinstance(o, dict):
            return {k: _rec(v) for k, v in o.items() if k != "not"}
        if isinstance(o, list):
            return [_rec(x) for x in o]
        return o
    return _rec(copy.deepcopy(obj))

def generer_instance_brute(schema_text, max_input_tokens=MAX_INPUT_TOKENS, max_new_tokens=MAX_NEW_TOKENS):
    prompt = INSTRUCTION + f"\nSchema:\n{schema_text}\nInstance:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_tok = inputs["input_ids"].shape[1]
    if n_tok > max_input_tokens:
        return None, "overflow"
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    genere = tokenizer.decode(out[0][n_tok:], skip_special_tokens=True)
    cand = extraire_premier_json(genere)
    if cand is None:
        return None, "tronqué"
    try:
        return json.loads(cand), None
    except json.JSONDecodeError:
        return None, "mal formé"

def valide_contre(instance, schema):
    if instance is None:
        return False
    try:
        jsonschema.validate(instance, schema)
        return True
    except jsonschema.ValidationError:
        return False

# test sur le même schéma que tout à l'heure
schema_sans = retirer_not(schema_test)
inst_avec, _ = generer_instance_brute(json.dumps(schema_test))
inst_sans, _ = generer_instance_brute(json.dumps(schema_sans))

print("avec not, valide :", valide_contre(inst_avec, schema_test))
print("sans not, valide :", valide_contre(inst_sans, schema_sans))

## 9. RAG — nécessite ta base de référence (`base_rag.csv`)

uploade `base_rag.csv` dans `/content/` avant de lancer cette section
(colonnes attendues : `nom_schema`, `schema`, `instance`).

In [ ]:
!pip install -q langchain langchain-community faiss-cpu sentence-transformers

In [ ]:
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

df = pd.read_csv("/content/base_rag.csv", keep_default_na=False)
df = df[df["instance"].str.strip() != "insatisfiable"].reset_index(drop=True)
print(f"{len(df)} schémas chargés (hors insatisfiables)")

noms = df["nom_schema"].tolist()
textes_schemas = df["schema"].tolist()
instances = df["instance"].tolist()

documents = [
    Document(page_content=row["schema"], metadata={"nom_schema": row["nom_schema"], "instance": row["instance"]})
    for _, row in df.iterrows()
]

embedding_code = HuggingFaceEmbeddings(model_name="microsoft/codebert-base")
base_vectorielle_code = FAISS.from_documents(documents, embedding_code)
print("base vectorielle codebert construite")

## 10. pipeline RAG complet, test sur un schéma

In [ ]:
from langchain_core.prompts import PromptTemplate

PROMPT_NEUTRE = """Tu es un générateur spécialisé en JSON. Ta tâche est de produire UNE SEULE instance JSON qui satisfait strictement le schéma JSON Schema fourni à la fin.

Voici des exemples de paires (schéma, instance valide) pour t'aider à comprendre le format attendu :

{exemples_formates}

Maintenant, génère une instance JSON valide pour CE schéma :

Schéma :
{schema_cible}

Réponds UNIQUEMENT avec l'instance JSON, sans aucune explication, sans balises de code, rien d'autre que le JSON.

Instance :
"""

prompt_template = PromptTemplate(input_variables=["exemples_formates", "schema_cible"], template=PROMPT_NEUTRE)

def recuperer_exemples(schema_cible_texte, base_vectorielle, k=1, exclure_identique=True):
    resultats = base_vectorielle.similarity_search(schema_cible_texte, k=k+1)
    exemples = []
    for doc in resultats:
        if exclure_identique and doc.page_content.strip() == schema_cible_texte.strip():
            continue
        exemples.append({"nom_schema": doc.metadata["nom_schema"], "schema": doc.page_content, "instance": doc.metadata["instance"]})
    return exemples[:k]

def formater_exemples(exemples):
    return "\n".join(f"Schéma:\n{ex['schema']}\nInstance:\n{ex['instance']}\n" for ex in exemples)

def valider_instance(instance_texte, schema_texte):
    try:
        instance = json.loads(instance_texte)
    except json.JSONDecodeError as e:
        return False, f"json mal formé : {e}"
    try:
        schema = json.loads(schema_texte)
        jsonschema.validate(instance, schema)
        return True, None
    except jsonschema.ValidationError as e:
        return False, f"invalide : {e.message}"

def pipeline_rag(nom_schema_cible, schema_cible_texte, base_vectorielle, k=1):
    exemples = recuperer_exemples(schema_cible_texte, base_vectorielle, k=k)
    prompt_final = prompt_template.format(exemples_formates=formater_exemples(exemples), schema_cible=schema_cible_texte)

    inputs = tokenizer(prompt_final, return_tensors="pt").to(model.device)
    n_tok = inputs["input_ids"].shape[1]
    with torch.no_grad():
        sortie = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1, pad_token_id=tokenizer.eos_token_id)
    brut = tokenizer.decode(sortie[0][n_tok:], skip_special_tokens=True)
    candidat = extraire_premier_json(brut)

    if candidat is None:
        return {"nom_schema": nom_schema_cible, "valide": False, "erreur": "tronqué/aucun json", "instance": None}
    valide, erreur = valider_instance(candidat, schema_cible_texte)
    return {"nom_schema": nom_schema_cible, "valide": valide, "erreur": erreur, "instance": candidat}

# test sur le premier schéma de la base
resultat = pipeline_rag(noms[0], textes_schemas[0], base_vectorielle_code, k=1)
print(resultat)

## 11. évaluer sur un échantillon (comparer les configurations)

reprends `rag/evaluate.py` et `rag/ensemble.py` du dépôt pour comparer
plusieurs valeurs de k, plusieurs embeddings, plusieurs prompts sur un
échantillon plus large.